**Generative AI Use**: For the purposes of the assignments, the use of generative AI is subject to the same policies regarding collaboration. Just as with other collaborators, each student must write down the solutions independently of the output of the interaction and the submission should include a note denoting the nature of the collaboration. The use of generative AI tools to substantially complete sections of the assignments is not in line with the spirit of the assignments, and would be a violation of the [Honor Code](https://communitystandards.stanford.edu/policies-and-guidance/honor-code).


---

**中文对照（参考译文，不含解答）**

**生成式 AI 的使用**：本作业对生成式 AI 的使用采用与协作相同的规则。与其他协作者交流之后，每位学生仍须独立写出自己的解答，不能直接依赖交流中得到的输出；提交时应附上说明，如实记录协作的性质。使用生成式 AI 实质性地完成作业的大部分内容，不符合本作业的教学目的，也会违反[学术诚信准则](https://communitystandards.stanford.edu/policies-and-guidance/honor-code)。

In [ ]:
# This mounts your Google Drive to the Colab VM.
# [中文] 这把Google车开到Colab VM
from google.colab import drive
drive.mount('/content/drive')

# TODO: Enter the foldername in your Drive where you have saved the unzipped
# assignment folder, e.g. 'cs231n/assignments/assignment3/'
# [中文] TODO: 在您的驱动器中输入文件夹名, 在那里您保存了未覆盖的任务文件夹e.g。 “ cs231n/ assignations/
# formantions3/ ”
FOLDERNAME = "cs231n/assignments/assignment3/"
assert FOLDERNAME is not None, "[!] Enter the foldername."

# Now that we've mounted your Drive, this ensures that
# the Python interpreter of the Colab VM can load
# python files from within it.
# [中文] 现在,我们已经安装了你的驱动器, 这确保了Python 翻译 科拉布VM 可以装载
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

# This downloads the COCO dataset to your Drive
# if it doesn't already exist.
# [中文] 如果COCO数据不存在, 它会下载到驱动器上。
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/datasets/
!bash get_datasets.sh
%cd /content/drive/My\ Drive/$FOLDERNAME

## Using GPU

Go to `Runtime > Change runtime type` and set `Hardware accelerator` to `GPU`. This will reset Colab. **Rerun the top cell to mount your Drive again.**

---

**中文对照（参考译文，不含解答）**

## 使用 GPU

转到 `Runtime > Change runtime type` 并设置 `Hardware accelerator` 到 `GPU`。 这将重置 Colab。 ** 重新运行顶部单元格以再次挂载您的驱动器。 **

# Self-Supervised Learning

## What is self-supervised learning?
Modern day machine learning requires lots of labeled data. But often times it's challenging and/or expensive to obtain large amounts of human-labeled data. Is there a way we could ask machines to automatically learn a model which can generate good visual representations without a labeled dataset? Yes, enter self-supervised learning!

Self-supervised learning (SSL) allows models to automatically learn a "good" representation space using the data in a given dataset without the need for their labels. Specifically, if our dataset were a bunch of images, then self-supervised learning allows a model to learn and generate a "good" representation vector for images.

The reason SSL methods have seen a surge in popularity is because the learnt model continues to perform well on other datasets as well i.e. new datasets on which the model was not trained on!

## What makes a "good" representation?
A "good" representation vector needs to capture the important features of the image as it relates to the rest of the dataset. This means that images in the dataset representing semantically similar entities should have similar representation vectors, and different  images in the dataset should have different representation vectors. For example, two images of an apple should have similar representation vectors, while an image of an apple and an image of a banana should have different representation vectors.

## Contrastive Learning: SimCLR
Recently, [SimCLR](https://arxiv.org/pdf/2002.05709.pdf) introduces a new architecture which uses **contrastive learning** to learn good visual representations. Contrastive learning aims to learn similar representations for similar images and different representations for different images. As we will see in this notebook, this simple idea allows us to train a surprisingly good model without using any labels.

Specifically, for each image in the dataset, SimCLR generates two differently augmented views of that image, called a **positive pair**. Then, the model is encouraged to generate similar representation vectors for this pair of images. See below for an illustration of the architecture (Figure 2 from the paper).

---

**中文对照（参考译文，不含解答）**

# 自监督学习

## 什么是自我监督的学习?
现代日间机器学习需要大量标签数据。 但获取大量人类标签数据往往具有挑战性和(或)昂贵。 我们能否要求机器自动学习一个模型,在没有标签数据集的情况下产生良好的视觉表现? 是的, 输入自我监督的学习 !

自监管学习 (SSL) 使模型能够自动学习一个“ 好” 的表达空间, 使用特定数据集中的数据而不需要标签。 具体而言, 如果我们的数据集是一堆图像, 那么自监管学习可以让模型学习并生成图像的“ 好”表达矢量。

SSL方法出现受欢迎程度剧增的原因是,所学模型在其他数据集 i.e 上仍然表现良好。 新的数据集没有经过该模型的训练!

## 是什么让"好"代表?
“ 好” 表示矢量需要捕捉图像与数据集其他部分有关的重要特征。 这意味着代表语义相似实体的数据集中的图像应该有类似的表达矢量,而数据集中的不同图像应该有不同的表达矢量。 例如,苹果的两个图像应该有相似的表达矢量;而苹果和香蕉的图像应该有不同表达矢量。

## 反向学习: SimCLR
最近, [SimCLR](https://arxiv.org/pdf/2002.05709.pdf) 引入了一个新的架构, 它使用** contractical learning** 来学习良好的视觉表现。 对比性学习旨在学习类似图像的类似表达方式和不同图像的不同表达方式。 正如我们从这个笔记本上可以看到的, 这个简单的想法使我们能够在不使用任何标签的情况下训练出一个惊人的好模型。

具体地说,对于数据集中的每张图像, SimCLR 生成了两种不同增强的图像视图, 称为 ** 阳性对**。 然后, 鼓励模型为这副图像生成相似的表达矢量。 请参见下面的架构图解 (Figure 2 from the paper)。

In [ ]:
# Run this cell to view the SimCLR architecture.
# [中文] 运行此单元格查看 SimCLR 结构。
from IPython.display import Image
Image('images/simclr_fig2.png', width=500)

Given an image **x**, SimCLR uses two different data augmentation schemes **t** and **t'** to generate the positive pair of images **$\tilde{x}_i$** and **$\tilde{x}_j$**. $f$ is a basic encoder net that  extracts representation vectors from the augmented data samples, which yields **$h_i$** and **$h_j$**, respectively. Finally, a small neural network projection head $g$ maps the representation vectors to the space where the contrastive loss is applied. The goal of the contrastive loss is to maximize agreement between the final vectors **$z_i = g(h_i)$** and **$z_j = g(h_j)$**. We will discuss the contrastive loss in more detail later, and you will get to implement it.

After training is completed, we throw away the projection head $g$ and only use $f$ and the representation $h$ to perform downstream tasks, such as classification. You will get a chance to finetune a layer on top of a trained SimCLR model for a classification task and compare its performance with a baseline model (without self-supervised learning).

---

**中文对照（参考译文，不含解答）**

鉴于图像**x**, SimCLR 使用两种不同的数据增强方案** t** 和 ** t** 来生成正对图像** $\tilde{x}_i$ ** 和 ** $\tilde{x}_j$ **。 $f$ 是一个基本的编码网,它从增强的数据样本中提取代表矢量,分别生成** $h_i$** 和** $h_j$ **。最后,一个小型神经网络投影头 $g$ 将表示矢量映射到应用对比性损失的空间。对比损失的目的是最大限度地实现最终矢量之间的一致。** $z_i = g(h_i)$ **和**$z_j = g(h_j)$**。我们稍后将更详细地讨论对比性损失,您将进入此位置。 执行。

训练完成后, 我们丢弃投影头 $g$, 只能使用 $f$ 和表示 $h$ 来完成下游任务, 如分类等。 您将有机会微调经过训练的 SimCLR 模型的一层, 用于分类任务, 并将其性能与基线模型进行比较( 不进行自我监督学习 )。

## Pretrained Weights
For your convenience, we have given you pretrained weights (trained for ~18 hours on CIFAR-10) for the SimCLR model. Run the following cell to download pretrained model weights to be used later. (This will take ~1 minute)

---

**中文对照（参考译文，不含解答）**

## 预训练权重
为方便起见,我们为SimCLR模型提供了预先训练的权重(在CIFAR- 10 上训练时间为 18 小时)。运行以下单元格下载预训练的权重,供日后使用。 (这将花费时间~ 1 分钟)

In [ ]:
%%bash
DIR=pretrained_model/
if [ ! -d "$DIR" ]; then
    mkdir "$DIR"
fi

URL=http://downloads.cs.stanford.edu/downloads/cs231n/pretrained_simclr_model.pth
# Try this if above doesn't work.
# URL=http://cs231n.stanford.edu/2025/storage/a3/pretrained_simclr_model.pth
# [中文] 如果以上不工作, 请尝试此选项。 URL=
# http://cs231n.stanford.edu/2025/storage/a3/pretrained_simclr_model.pth
FILE=pretrained_model/pretrained_simclr_model.pth
if [ ! -f "$FILE" ]; then
    echo "Downloading weights..."
    wget "$URL" -O "$FILE"
fi

In [ ]:
# Setup cell.
# [中文] 设置牢房。
%pip install thop
import torch
import os
import sys
import types
import importlib
import pandas as pd
import numpy as np
import torch.optim as optim
import torch.nn as nn
import random
from thop import profile, clever_format
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
import matplotlib.pyplot as plt
%matplotlib inline

if "imp" not in sys.modules:
    imp = types.ModuleType("imp")
    imp.reload = importlib.reload
    sys.modules["imp"] = imp

%load_ext autoreload
%autoreload 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Data Augmentation

Our first step is to perform data augmentation. Implement the `compute_train_transform()` function in `cs231n/simclr/data_utils.py` to apply the following random transformations:

1. Randomly resize and crop to 32x32.
2. Horizontally flip the image with probability 0.5
3. With a probability of 0.8, apply color jitter (see `compute_train_transform()` for definition)
4. With a probability of 0.2, convert the image to grayscale

---

**中文对照（参考译文，不含解答）**

# 数据增强

我们的第一步是执行数据增强。在 `cs231n/simclr/data_utils.py` 中执行 `compute_train_transform()` 函数以应用以下随机转换:

 1. 随机调整大小和裁剪到 32x32。
 2。 水平翻转图像, 概率为 0.5
 3. 概率为 0.8 时, 应用颜色抖动( 定义请参见 `compute_train_transform()` )
 4. 概率为 0.2 的图像转换为灰度

Now complete `compute_train_transform()` and `CIFAR10Pair.__getitem__()` in `cs231n/simclr/data_utils.py` to apply the data augmentation transform and generate **$\tilde{x}_i$** and **$\tilde{x}_j$**.



---

**中文对照（参考译文，不含解答）**

现在在 `cs231n/simclr/data_utils.py` 中完成 `compute_train_transform()` 和 `CIFAR10Pair.__getitem__()` 以应用数据增强变换并生成 ** $\tilde{x}_i$ ** 和 ** $\tilde{x}_j$ **。



Test to make sure that your data augmentation code is correct:

---

**中文对照（参考译文，不含解答）**

测试以确保您的数据增强代码正确:

In [ ]:
from cs231n.simclr.data_utils import *
from cs231n.simclr.contrastive_loss import *

answers = torch.load('simclr_sanity_check.key')

In [ ]:
from PIL import Image
import torchvision
from torchvision.datasets import CIFAR10

def test_data_augmentation(correct_output=None):
    train_transform = compute_train_transform(seed=2147483647)
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
    trainloader = torch.utils.data.DataLoader(trainset, batch_size=2, shuffle=False, num_workers=2)
    dataiter = iter(trainloader)
    images, labels = next(dataiter)
    img = torchvision.utils.make_grid(images)
    img = img / 2 + 0.5     # unnormalize
    # [中文] 无法正常化@ info: whatsthis
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()
    output = images

    print("Maximum error in data augmentation: %g"%rel_error( output.numpy(), correct_output.numpy()))

# Should be less than 1e-07.
# [中文] 应小于 1e-07。
test_data_augmentation(answers['data_augmentation'])

# Base Encoder and Projection Head
The next steps are to apply the base encoder and projection head to the augmented samples **$\tilde{x}_i$** and **$\tilde{x}_j$**.

The base encoder $f$ extracts representation vectors for the augmented samples. The SimCLR paper found that using deeper and wider models improved performance and thus chose [ResNet](https://arxiv.org/pdf/1512.03385.pdf) to use as the base encoder. The output of the base encoder are the representation vectors **$h_i = f(\tilde{x}_i$)** and **$h_j = f(\tilde{x}_j$)**.

The projection head $g$ is a small neural network that maps the representation vectors **$h_i$** and **$h_j$** to the space where the contrastive loss is applied. The paper found that using a nonlinear projection head improved the representation quality of the layer before it. Specifically, they used a MLP with one hidden layer as the projection head $g$. The contrastive loss is then computed based on the outputs **$z_i = g(h_i$)** and **$z_j = g(h_j$)**.

We provide implementations of these two parts in `cs231n/simclr/model.py`. Please skim through the file and make sure you understand the implementation.

---

**中文对照（参考译文，不含解答）**

# 基础编码器和投影头
下一步是将基准编码器和投影头应用到增强样本中** $\tilde{x}_i$**和** $\tilde{x}_j$**。

基编码器 $f$ 提取扩增样本的表示矢量。 SimCLR 论文发现,使用更深、更宽的模型可以提高性能,因此选择了 [ResNet](https://arxiv.org/pdf/1512.03385.pdf) 作为基编码器。基编码器的输出是表示矢量** $h_i = f(\tilde{x}_i$ ** 和 ** $h_j = f(\tilde{x}_j$ **。

投影头 $g$ 是一个小神经网络,它映射代表矢量** $h_i$ ** 和 ** $h_j$ ** 用于应用对比性损失的空间。 论文发现,使用非线性投影头提高了它面前层的显示质量。 具体地说, 它们使用一个隐藏层的 MLP 作为投影头$g$。 然后根据输出值** $z_i = g(h_i$ ** 和** $z_j = g(h_j$ 计算对比性损失。 **

我们在`cs231n/simclr/model.py` 中提供这两个部分的落实情况。 请略过文件, 并确保您了解执行情况。

# SimCLR: Contrastive Loss

A mini-batch of $N$ training images yields a total of $2N$ data-augmented examples. For each positive pair $(i, j)$ of augmented examples, the contrastive loss function aims to maximize the agreement of vectors $z_i$ and $z_j$. Specifically, the loss is the normalized temperature-scaled cross entropy loss and aims to maximize the agreement of $z_i$ and $z_j$ relative to all other augmented examples in the batch:

---

**中文对照（参考译文，不含解答）**

# SimCLR: 对抗性损失

一小批货$N$训练图象产生的总数$2N$数据放大示例。 对于每对正对$(i, j)$对比性损失功能旨在最大限度地实现矢量的一致$z_i$和$z_j$具体地说,损失是正常温度标定的交叉星体损耗,目的是最大限度地实现$z_i$和$z_j$相对于批次中所有其他增强示例:

$$
l \; (i, j) = -\log \frac{\exp (\;\text{sim}(z_i, z_j)\; / \;\tau) }{\sum_{k=1}^{2N} \mathbb{1}_{k \neq i} \exp (\;\text{sim} (z_i, z_k) \;/ \;\tau) }
$$

where $\mathbb{1} \in \{0, 1\}$ is an indicator function that outputs $1$ if $k\neq i$ and $0$ otherwise. $\tau$ is a temperature parameter that determines how fast the exponentials increase.

sim$(z_i, z_j) = \frac{z_i \cdot z_j}{|| z_i || || z_j ||}$ is the (normalized) dot product between vectors $z_i$ and $z_j$. The higher the similarity between $z_i$ and $z_j$, the larger the dot product is, and the larger the numerator becomes. The denominator normalizes the value by summing across $z_i$ and all other augmented examples $k$ in the batch. The range of the normalized value is $(0, 1)$, where a high score close to $1$ corresponds to a high similarity between the positive pair $(i, j)$ and low similarity between $i$ and other augmented examples $k$ in the batch. The negative log then maps the range $(0, 1)$ to the loss values $(\inf, 0)$.

The total loss is computed across all positive pairs $(i, j)$ in the batch. Let $z = [z_1, z_2, ..., z_{2N}]$ include all the augmented examples in the batch, where $z_{1}...z_{N}$ are outputs of the left branch, and $z_{N+1}...z_{2N}$ are outputs of the right branch. Thus, the positive pairs are $(z_{k}, z_{k + N})$ for $\forall k \in [1, N]$.

Then, the total loss $L$ is:

---

**中文对照（参考译文，不含解答）**

点$\mathbb{1} \in \{0, 1\}$是一个指标函数,即产出$1$如果$k\neq i$和$0$换句话说$\tau$是一个温度参数,它决定指数增加的速度。

模拟$(z_i, z_j) = \frac{z_i \cdot z_j}{|| z_i || || z_j ||}$是(normalized)矢量之间的点产物$z_i$和$z_j$. 两种情况之间的相似程度越高。$z_i$和$z_j$,圆点产品越大,分子就越大。分母通过对齐$z_i$和所有其他增加的例子$k$在批量中。归正值的范围是$(0, 1)$,高分接近$1$对应正对的高度相似性$(i, j)$和低相似性$i$和其他补充实例$k$在批量中。 负日志然后映射范围$(0, 1)$B. 对损失价值的调整$(\inf, 0)$ .

所有正对均计算总损失$(i, j)$在批量中。$z = [z_1, z_2, ..., z_{2N}]$包括批量中的所有增强示例,其中$z_{1}...z_{N}$是左侧分支的产出,以及$z_{N+1}...z_{2N}$是右分支的产出。 因此,正对是$(z_{k}, z_{k + N})$用于$\forall k \in [1, N]$ .

然后,总损失$L$是:

$$
L = \frac{1}{2N} \sum_{k=1}^N [ \; l(k, \;k+N) + l(k+N, \;k)\;]
$$

**NOTE:** this equation is slightly different from the one in the paper. We've rearranged the ordering of the positive pairs in the batch, so the indices are different. The rearrangement makes it easier to implement the code in vectorized form.

We'll walk through the steps of implementing the loss function in vectorized form. Implement the functions `sim`, `simclr_loss_naive` in `cs231n/simclr/contrastive_loss.py`. Test your code by running the sanity checks below.

---

**中文对照（参考译文，不含解答）**

** 此方程式与论文中的方程式略有不同。 我们已经重新排列了批次中正对的顺序, 因此指数是不同的。 重新排列使得以矢量化形式执行代码更加容易。

我们将以矢量化形式执行损失函数的步骤。 在 `cs231n/simclr/contrastive_loss.py` 中执行函数 `sim`, `simclr_loss_naive`。 在下面运行理智检查来测试您的代码。

In [ ]:
from cs231n.simclr.contrastive_loss import *
answers = torch.load('simclr_sanity_check.key')

In [ ]:
def test_sim(left_vec, right_vec, correct_output):
    output = sim(left_vec, right_vec).cpu().numpy()
    print("Maximum error in sim: %g"%rel_error(correct_output.numpy(), output))

# Should be less than 1e-07.
# [中文] 应小于 1e-07。
test_sim(answers['left'][0], answers['right'][0], answers['sim'][0])
test_sim(answers['left'][1], answers['right'][1], answers['sim'][1])

In [ ]:
def test_loss_naive(left, right, tau, correct_output):
    naive_loss = simclr_loss_naive(left, right, tau).item()
    print("Maximum error in simclr_loss_naive: %g"%rel_error(correct_output, naive_loss))

# Should be less than 1e-07.
# [中文] 应小于 1e-07。
test_loss_naive(answers['left'], answers['right'], 5.0, answers['loss']['5.0'])
test_loss_naive(answers['left'], answers['right'], 1.0, answers['loss']['1.0'])

Now implement the vectorized version by implementing `sim_positive_pairs`, `compute_sim_matrix`, `simclr_loss_vectorized` in `cs231n/simclr/contrastive_loss.py`.  Test your code by running the sanity checks below.

---

**中文对照（参考译文，不含解答）**

现在通过在 `cs231n/simclr/contrastive_loss.py` 中执行 `sim_positive_pairs` 、 `compute_sim_matrix` 和 `simclr_loss_vectorized` 来执行矢量化版本。 在下面运行正常状态检查来测试您的代码。

In [ ]:
def test_sim_positive_pairs(left, right, correct_output):
    sim_pair = sim_positive_pairs(left, right).cpu().numpy()
    print("Maximum error in sim_positive_pairs: %g"%rel_error(correct_output.numpy(), sim_pair))

# Should be less than 1e-07.
# [中文] 应小于 1e-07。
test_sim_positive_pairs(answers['left'], answers['right'], answers['sim'])

In [ ]:
def test_sim_matrix(left, right, correct_output):
    out = torch.cat([left, right], dim=0)
    sim_matrix = compute_sim_matrix(out).cpu()
    assert torch.isclose(sim_matrix, correct_output).all(), "correct: {}. got: {}".format(correct_output, sim_matrix)
    print("Test passed!")

test_sim_matrix(answers['left'], answers['right'], answers['sim_matrix'])

In [ ]:
def test_loss_vectorized(left, right, tau, correct_output):
    vec_loss = simclr_loss_vectorized(left, right, tau, device=left.device).item()
    print("Maximum error in loss_vectorized: %g"%rel_error(correct_output, vec_loss))

# Should be less than 1e-07.
# [中文] 应小于 1e-07。
test_loss_vectorized(answers['left'], answers['right'], 5.0, answers['loss']['5.0'])
test_loss_vectorized(answers['left'], answers['right'], 1.0, answers['loss']['1.0'])

# Implement the train function
Complete the `train()` function in `cs231n/simclr/utils.py` to obtain the model's output and use `simclr_loss_vectorized` to compute the loss. (Please take a look at the `Model` class in `cs231n/simclr/model.py` to understand the model pipeline and the returned values)

---

**中文对照（参考译文，不含解答）**

# 执行火车功能
在 `cs231n/simclr/utils.py` 中完成 `train()` 函数, 以获取模型输出, 并使用 `simclr_loss_vectorized` 来计算损失。 (请查看 `cs231n/simclr/model.py` 中的 `Model` 类, 以了解模型管道和返回值)

In [ ]:
from cs231n.simclr.data_utils import *
from cs231n.simclr.model import *
from cs231n.simclr.utils import *

### Train the SimCLR model

Run the following cells to load in the pretrained weights and continue to train a little bit more. This part will take ~10 minutes and will output to `pretrained_model/trained_simclr_model.pth`.

**NOTE:** Don't worry about logs such as '_[WARN] Cannot find rule for ..._'. These are related to another module used in the notebook. You can verify the integrity of your code changes through our provided prompts and comments.

---

**中文对照（参考译文，不含解答）**

### 训练 SimCLR 模型

运行以下单元格, 以装入预设的权重, 并继续多加训练。 此部分需要 ~ 10 分钟, 输出到 `pretrained_model/trained_simclr_model.pth`。

** 不要为“_[WARN] 找不到规则......_” 等日志担心。 这些日志与笔记本中使用的另一个模块有关。 您可以通过我们提供的提示和批注来验证您的代码更改的完整性。

In [ ]:
# Do not modify this cell.
# [中文] 不修改此单元格。
feature_dim = 128
temperature = 0.5
k = 200
batch_size = 64
epochs = 1
temperature = 0.5
percentage = 0.5
pretrained_path = './pretrained_model/pretrained_simclr_model.pth'

# Prepare the data.
# [中文] 准备数据
train_transform = compute_train_transform()
train_data = CIFAR10Pair(root='data', train=True, transform=train_transform, download=True)
train_data = torch.utils.data.Subset(train_data, list(np.arange(int(len(train_data)*percentage))))
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=16, pin_memory=True, drop_last=True)
test_transform = compute_test_transform()
memory_data = CIFAR10Pair(root='data', train=True, transform=test_transform, download=True)
memory_loader = DataLoader(memory_data, batch_size=batch_size, shuffle=False, num_workers=16, pin_memory=True)
test_data = CIFAR10Pair(root='data', train=False, transform=test_transform, download=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=16, pin_memory=True)

# Set up the model and optimizer config.
# [中文] 设置模型和优化配置。
model = Model(feature_dim)
model.load_state_dict(torch.load(pretrained_path, map_location='cpu'), strict=False)
model = model.to(device)
flops, params = profile(model, inputs=(torch.randn(1, 3, 32, 32).to(device),))
flops, params = clever_format([flops, params])
print('# Model Params: {} FLOPs: {}'.format(params, flops))
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-6)
c = len(memory_data.classes)

# Training loop.
# [中文] 训练循环。
results = {'train_loss': [], 'test_acc@1': [], 'test_acc@5': []} #<< -- output
# [中文] -- -- 产出

if not os.path.exists('results'):
    os.mkdir('results')
best_acc = 0.0
for epoch in range(1, epochs + 1):
    train_loss = train(model, train_loader, optimizer, epoch, epochs, batch_size=batch_size, temperature=temperature, device=device)
    results['train_loss'].append(train_loss)
    test_acc_1, test_acc_5 = test(model, memory_loader, test_loader, epoch, epochs, c, k=k, temperature=temperature, device=device)
    results['test_acc@1'].append(test_acc_1)
    results['test_acc@5'].append(test_acc_5)

    # Save statistics.
    # [中文] 保存统计数据。
    if test_acc_1 > best_acc:
        best_acc = test_acc_1
        torch.save(model.state_dict(), './pretrained_model/trained_simclr_model.pth')

# Finetune a Linear Layer for Classification!

Now it's time to put the representation vectors to the test!

We remove the projection head from the SimCLR model and slap on a linear layer to finetune for a simple classification task. All layers before the linear layer are frozen, and only the weights in the final linear layer are trained. We compare the performance of the SimCLR + finetuning model against a baseline model, where no self-supervised learning is done beforehand, and all weights in the model are trained. You will get to see for yourself the power of self-supervised learning and how the learned representation vectors improve downstream task performance.


---

**中文对照（参考译文，不含解答）**

# 将线性图层精细配置为分类 !

现在是时候测试代表矢量了!

我们从 SimCLR 模型中移除投影头, 并拍打一个线性层, 以微调简单分类任务。 线性层之前的所有层都被冻结, 只有最后线性层的权重才经过训练。 我们比较SimCLLR + 微调模型的性能和基线模型, 基准模型之前没有进行自我监督的学习, 模型中的所有权重都经过训练。 您将会看到自我监督学习的力量, 以及所学的演示矢量如何改善下游任务性能。


## Baseline: Without Self-Supervised Learning
First, let's take a look at the baseline model. We'll remove the projection head from the SimCLR model and slap on a linear layer to finetune for a simple classification task. No self-supervised learning is done beforehand, and all weights in the model are trained. Run the following cells.

**NOTE:** Don't worry if you see low but reasonable performance.

---

**中文对照（参考译文，不含解答）**

## 基线:没有自学
首先,让我们看看基线模型。 我们将从 SimCLR 模型中移除投影头, 并拍打一个线性层来微调一个简单的分类任务。 事先没有进行自我监督的学习, 并且模型中的所有权重都经过了训练。 运行下面的单元格。

不要担心,如果你看到低但合理的性能。

In [ ]:
class Classifier(nn.Module):
    def __init__(self, num_class):
        super(Classifier, self).__init__()

        # Encoder.
        # [中文] 编码器
        self.f = Model().f

        # Classifier.
        # [中文] 分类。
        self.fc = nn.Linear(2048, num_class, bias=True)

    def forward(self, x):
        x = self.f(x)
        feature = torch.flatten(x, start_dim=1)
        out = self.fc(feature)
        return out

In [ ]:
# Do not modify this cell.
# [中文] 不修改此单元格。
feature_dim = 128
temperature = 0.5
k = 200
batch_size = 128
epochs = 10
percentage = 0.1

train_transform = compute_train_transform()
train_data = CIFAR10(root='data', train=True, transform=train_transform, download=True)
trainset = torch.utils.data.Subset(train_data, list(np.arange(int(len(train_data)*percentage))))
train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=16, pin_memory=True)
test_transform = compute_test_transform()
test_data = CIFAR10(root='data', train=False, transform=test_transform, download=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=16, pin_memory=True)

model = Classifier(num_class=len(train_data.classes)).to(device)
for param in model.f.parameters():
    param.requires_grad = False

flops, params = profile(model, inputs=(torch.randn(1, 3, 32, 32).to(device),))
flops, params = clever_format([flops, params])
print('# Model Params: {} FLOPs: {}'.format(params, flops))
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3, weight_decay=1e-6)
no_pretrain_results = {'train_loss': [], 'train_acc@1': [], 'train_acc@5': [],
           'test_loss': [], 'test_acc@1': [], 'test_acc@5': []}

best_acc = 0.0
for epoch in range(1, epochs + 1):
    train_loss, train_acc_1, train_acc_5 = train_val(model, train_loader, optimizer, epoch, epochs, device='cuda')
    no_pretrain_results['train_loss'].append(train_loss)
    no_pretrain_results['train_acc@1'].append(train_acc_1)
    no_pretrain_results['train_acc@5'].append(train_acc_5)
    test_loss, test_acc_1, test_acc_5 = train_val(model, test_loader, None, epoch, epochs)
    no_pretrain_results['test_loss'].append(test_loss)
    no_pretrain_results['test_acc@1'].append(test_acc_1)
    no_pretrain_results['test_acc@5'].append(test_acc_5)
    if test_acc_1 > best_acc:
        best_acc = test_acc_1

# Print the best test accuracy.
# [中文] 打印最佳测试准确率。
print('Best top-1 accuracy without self-supervised learning: ', best_acc)

## With Self-Supervised Learning

Let's see how much improvement we get with self-supervised learning. Here, we pretrain the SimCLR model using the simclr loss you wrote, remove the projection head from the SimCLR model, and use a linear layer to finetune for a simple classification task.

---

**中文对照（参考译文，不含解答）**

## 与自强学习

让我们看看自我监督的学习取得了多大的改进。 在这里, 我们使用您所写的 simpclr 损失来预设 SimCLR 模型, 从 SimCR 模型中移除投影头, 并使用线性层来微调一个简单的分类任务。

In [ ]:
# Do not modify this cell.
# [中文] 不修改此单元格。
feature_dim = 128
temperature = 0.5
k = 200
batch_size = 128
epochs = 10
percentage = 0.1
pretrained_path = './pretrained_model/trained_simclr_model.pth'

train_transform = compute_train_transform()
train_data = CIFAR10(root='data', train=True, transform=train_transform, download=True)
trainset = torch.utils.data.Subset(train_data, list(np.arange(int(len(train_data)*percentage))))
train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=16, pin_memory=True)
test_transform = compute_test_transform()
test_data = CIFAR10(root='data', train=False, transform=test_transform, download=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=16, pin_memory=True)

model = Classifier(num_class=len(train_data.classes))
model.load_state_dict(torch.load(pretrained_path, map_location='cpu'), strict=False)
model = model.to(device)
for param in model.f.parameters():
    param.requires_grad = False

flops, params = profile(model, inputs=(torch.randn(1, 3, 32, 32).to(device),))
flops, params = clever_format([flops, params])
print('# Model Params: {} FLOPs: {}'.format(params, flops))
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3, weight_decay=1e-6)
pretrain_results = {'train_loss': [], 'train_acc@1': [], 'train_acc@5': [],
           'test_loss': [], 'test_acc@1': [], 'test_acc@5': []}

best_acc = 0.0
for epoch in range(1, epochs + 1):
    train_loss, train_acc_1, train_acc_5 = train_val(model, train_loader, optimizer, epoch, epochs)
    pretrain_results['train_loss'].append(train_loss)
    pretrain_results['train_acc@1'].append(train_acc_1)
    pretrain_results['train_acc@5'].append(train_acc_5)
    test_loss, test_acc_1, test_acc_5 = train_val(model, test_loader, None, epoch, epochs)
    pretrain_results['test_loss'].append(test_loss)
    pretrain_results['test_acc@1'].append(test_acc_1)
    pretrain_results['test_acc@5'].append(test_acc_5)
    if test_acc_1 > best_acc:
        best_acc = test_acc_1

# Print the best test accuracy. You should see a best top-1 accuracy of >=70%.
# [中文] 打印最好的测试准确率。 您应该看到最好的顶部 --1的准确度 {_____________________________70% .
print('Best top-1 accuracy with self-supervised learning: ', best_acc)

### Plot your Comparison

Plot the test accuracies between the baseline model (no pretraining) and same model pretrained with self-supervised learning.

---

**中文对照（参考译文，不含解答）**

### 绘制您的比较

在基准模型 (no pretraining) 和经过自我监督学习的预先训练的同一模型之间绘制测试精度。

In [ ]:
plt.plot(no_pretrain_results['test_acc@1'], label="Without Pretrain")
plt.plot(pretrain_results['test_acc@1'], label="With Pretrain")
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Test Top-1 Accuracy')
plt.legend()
plt.show()